In [ ]:
import json
import os

import cmocean as cmo
import matplotlib.gridspec as gridspec
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.rcParams["font.family"] = "monospace"

In [ ]:
with open("../data/models_members.json", "r") as f:
    models_members = json.load(f)

In [ ]:
def create_metrics_df(metric_type, metrics_dir, models_members):
    """
    Loads and processes metrics data from JSON files for a specific metric type.

    Args:
        metric_type (str): The type of metric (e.g., 'proc', 'perf', 'tel').
        metrics_dir (str): The directory path containing the JSON files.
        models_members (dict): Dictionary of models and their members to process.

    Returns:
        pd.DataFrame: A DataFrame containing the processed metrics.
    """
    all_metrics_data = []

    for model, _members in models_members.items():
        for member in _members:
            filename = f"cmip6_historical_ENSO_{metric_type}_EnsoCMIP6regrid_{model}_{member}.json"
            metrics_path = os.path.join(metrics_dir, filename)

            if not os.path.exists(metrics_path):
                print(f"Warning: File not found, skipping. {metrics_path}")
                continue

            with open(metrics_path) as f:
                _metrics_data = json.load(f)

            metrics_data = _metrics_data["RESULTS"]["model"][model][member]["value"]
            _row_value = {"model": model, "member": member}

            for metric, data in metrics_data.items():
                _value = pd.DataFrame(data["metric"]).mean(axis=1)["value"]
                _row_value[metric] = _value

            all_metrics_data.append(_row_value)

    final_df = pd.DataFrame(all_metrics_data)
    if not final_df.empty:
        final_df = final_df.set_index(["model", "member"])

    return final_df

# ENSO metrics

In [ ]:
# Define your input directories
proc_metrics_dir = "../data/EnsoCMIP6regrid/ENSO_proc/"
perf_metrics_dir = "../data/EnsoCMIP6regrid/ENSO_perf/"
tel_metrics_dir = "../data/EnsoCMIP6regrid/ENSO_tel/"

proc_metrics_df = create_metrics_df("proc", proc_metrics_dir, models_members)
perf_metrics_df = create_metrics_df("perf", perf_metrics_dir, models_members)
tel_metrics_df = create_metrics_df("tel", tel_metrics_dir, models_members)

In [ ]:
proc_metrics_df.head()

In [ ]:
perf_metrics_df.head()

In [ ]:
tel_metrics_df.head()

In [ ]:
metrics_raw = pd.merge(
    pd.merge(
        proc_metrics_df,
        perf_metrics_df,
        on=["model", "member"],
        suffixes=("", "_right_drop"),
    ),
    tel_metrics_df,
    on=["model", "member"],
    suffixes=("", "_right_drop"),
)
cols_to_drop = [col for col in metrics_raw.columns if col.endswith("_right_drop")]
metrics_raw = metrics_raw.drop(columns=cols_to_drop)
metrics_raw

In [ ]:
metrics = metrics_raw.groupby("model").mean()

# Coastal metrics

In [ ]:
coastal_stats = pd.read_csv("../data/coastal_stats_esgf.csv")
coastal_stats

In [ ]:
coastal_obs_mean = coastal_stats.iloc[-2:, 2:].mean()
coastal_stats = coastal_stats.iloc[:-2, :]
coastal_stats_obs_mean = pd.DataFrame(coastal_obs_mean).T
coastal_stats_by_model = (
    coastal_stats.groupby("model")[coastal_stats.columns[2:]].mean().reset_index()
)
coastal_stats_by_model.head()

In [ ]:
# repeat coastal_stats_obs_mean by the number of members
coastal_stats_obs_mean = pd.concat(
    [coastal_stats_obs_mean] * len(coastal_stats_by_model), ignore_index=True
)

In [ ]:
coastal_stats_by_model_err = (
    coastal_stats_by_model.iloc[:, 1:] - coastal_stats_obs_mean
) / abs(coastal_stats_obs_mean)
coastal_stats_by_model_err["model"] = coastal_stats_by_model["model"]
coastal_stats_by_model_err = coastal_stats_by_model_err[coastal_stats_by_model.columns]
coastal_stats_by_model_err

In [ ]:
coastal_stats_by_model = coastal_stats_by_model_err

# GCMs on both datasets

In [ ]:
perf_models = metrics.copy().reset_index()
common_models = set(perf_models["model"].to_list()).intersection(
    set(coastal_stats_by_model["model"].to_list())
)
common_models = list(common_models)

common_perf_df = perf_models.query("model in @common_models").copy()
common_coa_df = coastal_stats_by_model.query("model in @common_models").copy()
for col in common_coa_df.columns[1:]:
    common_perf_df[col] = common_coa_df[col].values
common_perf_df["model_num"] = np.arange(len(common_perf_df))
common_perf_df = common_perf_df.dropna(axis=1)

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats


def get_annotation(r_val, p_val):
    """Formats the label, bolding it if significant."""
    if p_val < 0.05:
        return f"$\\mathbf{{{r_val:.2f}}}$"
    return f"{r_val:.2f}"


perf_cols = common_perf_df.columns[1:-4]
coa_cols = common_coa_df.columns[1:]

r_values = pd.DataFrame(index=perf_cols, columns=coa_cols, dtype=float)
p_values = pd.DataFrame(index=perf_cols, columns=coa_cols, dtype=float)

for x in perf_cols:
    for y in coa_cols:
        r, p = stats.pearsonr(common_perf_df[x], common_coa_df[y])
        r_values.loc[x, y] = r
        p_values.loc[x, y] = p
        # print(f"var1: {x}, var2: {y}, r: {r:.2f}, p-value: {p:.4f}")

name = "metrics"
r_values = (
    r_values.T.rename_axis("coastal_stats", axis=0)
    .T.rename_axis(name, axis=1)
    .sort_values(
        by=["coa_per_century", "en12_pr_std", "en34_std", "alpha"], ascending=False
    )
)

p_values = p_values.reindex(index=r_values.index, columns=r_values.columns)

In [ ]:
feedbacks_metrics = {
    "EnsoFbSstTaux": "SST-Taux",
    "EnsoFbTauxSsh": "Taux-SSH",
    "EnsoFbSshSst": "SSH-SST",
    "EnsoFbSstThf": "SST-NHF",
    "EnsodSstOce_2": "Oceanic processes\nleading to ENSO",
}

enso_metrics = {
    "EnsoAmpl": "Amplitude",
    "EnsoSeasonality": "Seasonal timing",
    "EnsoSstSkew": "Asymmetry",
    "EnsoDuration": "Duration",
    "EnsoSstDiversity_2": "Diversity",
    "EnsoSstLonRmse": "Pattern",  # zonal SSTA during ENSO
    "EnsoSstTsRmse": "Lifecycle",  # Temporal evolution of SSTA centered on ENSO
    "alpha": r"$\alpha-value$",
}

mean_seasonal = {
    "BiasPrLatRmse": "Double ITCZ",
    "BiasPrLonRmse": "Zonal mean PR",
    "BiasSstLonRmse": "Zonal mean SST",
    "BiasTauxLonRmse": "Zonal mean Taux",
    "SeasonalPrLatRmse": "Double ITCZ\nseasonal cycle",
    "SeasonalPrLonRmse": "Zonal mean PR\nseasonal cycle",
    "SeasonalSstLonRmse": "Zonal mean SST\nseasonal cycle",
    "SeasonalTauxLonRmse": "Zonal mean Taux\nseasonal cycle",
}

coa_metrics = {
    "coa_per_century": "COA",
    "standalone_coa_per_century": "Standalone COA",
    "nonstandalone_coa_per_century": "Spreading COA",
    "alpha": r"$\alpha-value$",
    "en12_std": r"$\sigma_{EN12}^{SST}$",
    "en34_std": r"$\sigma_{EN34}^{SST}$",
    "en12_pr_std": r"$\sigma_{EN12}^{PR}$",
}

In [ ]:
sort_cols = ["coa_per_century", "en12_pr_std", "en34_std", "alpha"]

first_table = r_values.loc[list(mean_seasonal.keys())].sort_values(
    by=sort_cols, ascending=False
)
second_table = r_values.loc[list(enso_metrics.keys())].sort_values(
    by=sort_cols, ascending=False
)
third_table = r_values.loc[list(feedbacks_metrics.keys())].sort_values(
    by=sort_cols, ascending=False
)

first_table_order = first_table.index
second_table_order = second_table.index
third_table_order = third_table.index

In [ ]:
fig = plt.figure(constrained_layout=True, figsize=(16, 10))
tables = [first_table.T, second_table.T, third_table.T]
labels = [mean_seasonal, enso_metrics, feedbacks_metrics]

xgroups = ["(a) Bias and seasonal cycle", "(b) ENSO properties", "(c) ENSO Feedbacks"]


ncols_list = [df.shape[1] for df in tables]
max_cols = sum(ncols_list)

gs = gridspec.GridSpec(nrows=2, ncols=max_cols, figure=fig, wspace=0.6)

hm_kwargs = dict(
    fmt="",
    cmap=cmo.cm.balance,
    linewidths=0.55,
    linecolor="black",
    square=True,
    vmin=-1,
    vmax=1,
    cbar=False,
)

for n, (ncols, _table, xlabel, xgroup) in enumerate(
    zip(ncols_list, tables, labels, xgroups)
):
    table = _table.copy()
    table[table == 1] = np.nan

    idx_start = sum(ncols_list[:n])
    ax = fig.add_subplot(gs[:, idx_start : idx_start + ncols])
    sns.heatmap(
        table,
        ax=ax,
        **hm_kwargs,
    )

    for i, y in enumerate(table.index):
        for j, x in enumerate(table.columns):
            if y == x:
                ax.add_patch(
                    patches.Rectangle(
                        (j, i),
                        1,
                        1,
                        hatch="///",
                        fill=True,
                        edgecolor="gray",
                        facecolor="white",
                    )
                )
                continue
            val = table.loc[y, x]
            p_val = p_values.T.loc[y, x]

            weight = "bold" if p_val < 0.05 else "normal"

            bg_color = ax.get_children()[0].get_facecolor()[i * table.shape[1] + j]
            color = (
                "white"
                if (bg_color[0] * 0.299 + bg_color[1] * 0.587 + bg_color[2] * 0.114)
                < 0.5
                else "black"
            )
            ax.text(
                j + 0.5,
                i + 0.5,
                f"{val:.2f}",
                ha="center",
                va="center",
                fontweight=weight,
                color=color,
                fontsize=12,
            )

    xtick_labels = [x.get_text() for x in ax.get_xticklabels()]
    ax.set_xticklabels([xlabel.get(x, x) for x in xtick_labels], size=14)

    if n != 0:
        ax.set_yticklabels([])
        ax.set_ylabel("")
    else:
        ytick_labels = [y.get_text() for y in ax.get_yticklabels()]
        ax.set_yticklabels([coa_metrics.get(y, y) for y in ytick_labels], size=14)

    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title(xgroup, fontsize=20)
    # axs.append(ax)
    for _, spine in ax.spines.items():
        spine.set_visible(True)
        # Use the same linewidth and color as the heatmap grid
        spine.set_linewidth(hm_kwargs["linewidths"])
        spine.set_color(hm_kwargs["linecolor"])